In [90]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import numpy as np
scaler = StandardScaler()
imputer = SimpleImputer(strategy = 'median')



In [73]:
#read player

player_crr_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_career_successes.csv')
player_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/players.csv')
# player_ply_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_play.csv')
print('read players')

read players


In [2]:
# read games

games_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/games.csv')
print('read games')

read games


In [3]:
#read games 2023
game_trk_2023_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2023.csv')
print('read games 2023')


read games 2023


In [4]:
#read games 2024
game_trk_2024_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2024.csv')
print('read games 2024')


read games 2024


In [5]:
#read games 2025
game_trk_2025_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2025.csv')
print('read games 2025')

read games 2025


In [80]:
#read combine_res
combine_res_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_results.csv')
print("read combine_res")

read combine_res


In [101]:
#read combine_trk
combine_trk_df = pd.read_csv('./nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_tracking.csv')
print("read combine_trk")


read combine_trk


In [112]:
#List of all current dfs
dfs = {'combine_res': combine_res_df, 'game_trk_2023': game_trk_2023_df, 'game_trk_2024': game_trk_2024_df, 
       'game_trk_2025': game_trk_2025_df, 'player_crr': player_crr_df, 'player': player_df, 
       'combine_trk': combine_trk_df, 'games': games_df}

#combine_res_df gives their positions and stats. these stats should be compared to others in same position. each position only appears once
#game_trk_202*_df gives speed info... still get their means and max/min from here use for later
#combine_trk_df gives drill type and drill name. separate for now and create a df where all their same drill names are grouped
#im going to ignore x, y, dir, attempt, (maybe dis), draft_year for now in combine_trk_df 
game_trk_2024_df.head()

,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2024080952,3379,57367,2024-08-10T01:30:31.800,87.09,22.52,0.28,0.11,0.03,98.51,126.08,NaN
1,2024080952,3379,57367,2024-08-10T01:30:31.900,87.11,22.51,0.24,0.18,0.02,97.33,125.81,NaN
2,2024080952,3379,57367,2024-08-10T01:30:32.000,87.12,22.50,0.17,0.24,0.01,96.65,124.99,NaN
3,2024080952,3379,57367,2024-08-10T01:30:32.100,87.13,22.49,0.15,0.21,0.02,95.42,120.75,NaN
4,2024080952,3379,57367,2024-08-10T01:30:32.200,87.14,22.48,0.11,0.19,0.01,95.42,136.32,NaN


In [103]:
#=========================================== Begin combine_trk_df normalization======================
combine_trk_df = combine_trk_df[
    combine_trk_df['entity_type'] == 'PLAYER'
    ]

In [104]:
combine_trk_df = combine_trk_df.drop(
    columns = ['x', 'y', 'dis', 'dir', 'draft_year', 'entity_type']
)

In [105]:

combine_trk_df = combine_trk_df.groupby(
    ['drill_type', 'drill_name', 'nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    min_acceleration=('a','min'),
    total_frames = ('s','count')
).reset_index()

In [106]:
scale_columns = ['max_speed', 'avg_speed', 
                 'max_acceleration', 'avg_acceleration', 
                 'min_acceleration']
for col in scale_columns:
    combine_trk_df[col] = combine_trk_df.groupby(['drill_type', 'drill_name'])[col].transform(
        lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).flatten()
    )

#all rows are normalized for their specific drill type and drill name. (at least should be...)

#=========================================== End combine_trk_df normalization======================

In [58]:

#=========================================norm combine_res_dfs======================================
combine_res_dfs_sep = {
    value: df for value, df in dfs['combine_res'].groupby('combine_position')
}

In [61]:



scale_columns = ['combine_height', 'combine_weight', 'hand_size', 
                 'arm_length', 'wing_span', 'ten_yd_split', 'forty', 
                 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps', 
                 'ngs_athleticism_score', 'ngs_college_production_score', 'ngs_final_score']

ct = ColumnTransformer(
        transformers=[
            ('num_scaler', StandardScaler(), scale_columns)
        ],
        remainder='passthrough'
    )
ct.set_output(transform='pandas')

for key, val in combine_res_dfs_sep.items():
    cols_with_nan = val.columns[
        val.isna().any()
    ].tolist()
    
    val[cols_with_nan] = imputer.fit_transform(
        val[cols_with_nan]
    )
    
    val[scale_columns] = scaler.fit_transform(
        val[scale_columns]
    )
    combine_res_dfs_sep[key] = val

#now all have z scores dependent on their position 
#removed all nans too prior to z scores with imputation (replace with median)

In [62]:
merged_aggr_combine_res_df = pd.concat(
    combine_res_dfs_sep, axis=0, ignore_index=True
)

#concat/merge all combine dfs with norm vals

In [67]:
#===========================================end norm combine_res_dfs====================================

In [114]:

#==========================================================Begin Cleaning game_[year] ===============================================================
aggr_games_trk2025_df = dfs['game_trk_2025'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()


In [115]:
aggr_games_trk2024_df = dfs['game_trk_2024'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()


In [116]:
aggr_games_trk2023_df = dfs['game_trk_2023'].groupby(
    ['nfl_id']).agg(
    max_speed=('s','max'),
    avg_speed=('s', 'mean'),
    max_acceleration=('a','max'),
    avg_acceleration=('a','mean'),
    total_frames=('s','count')
).reset_index()

                  

In [118]:
all_aggr_games_trk_df = pd.concat(
    [aggr_games_trk2023_df, aggr_games_trk2024_df, aggr_games_trk2025_df], 
    ignore_index = True
)
all_aggr_games_trk_df = all_aggr_games_trk_df.groupby('nfl_id', as_index=False).agg(
    max_speed = ('max_speed', 'max'),
    avg_speed=('avg_speed', 'mean'),
    max_acceleration=('max_acceleration','max'),
    avg_acceleration=('avg_acceleration','mean'),
    total_frames=('total_frames','sum')
).reset_index()

scale_columns = ['max_speed', 'avg_speed', 
                 'max_acceleration', 'avg_acceleration']

all_aggr_games_trk_df[scale_columns] = scaler.fit_transform(
    all_aggr_games_trk_df[scale_columns]
)

#=============================================================================END cleaning game_[year]=================================================

In [133]:
#checkpoint before encoding
#combine_res_df -> merged_aggr_combine_res_df
#game_trk_2025_df, game_trk_2024_df, game_trk_2023_df -> all_aggr_games_trk_df
#combine_trk_df -> combine_trk_df (but cleaned)
#all left is player, player_crr, and game_df

progress_so_far = [merged_aggr_combine_res_df, all_aggr_games_trk_df, combine_trk_df]
player_crr_df.head()
#grab age of player, college, conference, 
#draft_round, draft_pick_within_round, draft_overall_pick from player_df

#grab career_offensive_snaps, career_defensive_snaps

,nfl_id,career_offensive_snaps,career_defensive_snaps,career_special_teams_snaps,career_games_active,career_games_started,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot
0,55867,0,1884,92,51,44,1,0,1
1,55870,2802,0,169,47,43,0,0,0
2,55874,3222,0,167,51,49,0,1,0
3,55875,3011,0,170,51,48,0,0,0
4,55877,0,1056,387,51,2,0,0,0


In [13]:
# dfs['player_ply'].loc[(dfs['player_ply']['game_id'] == 2025111610) & (dfs['player_ply']['play_id'] == 2279)]
# '(8:42) (Shotgun) L.Jackson pass incomplete deep left to D.Hopkins.'
# print()
# dfs['player'].loc[dfs['player']['nfl_id'] == 57183] -> Roger Rosengarten

# player_play csv seems far too complicated for ml project

In [14]:
#======================================================================Begin encoding==========================================
aggr_combine_trk_df = pd.get_dummies(aggr_combine_trk_df, columns = ['drill_type'], dtype=int)

In [15]:
player_df = pd.get_dummies(player_df, columns = ['nfl_position', 'college_name', 'college_conference'], dtype=int)

In [16]:
player_df = player_df.drop(columns = 'display_name')

In [17]:
player_df['Age'] = ((pd.to_datetime(player_df['draft_year'], format='%Y') - pd.to_datetime(player_df['birth_date'])).dt.days / 365.25).astype(int)

In [18]:
player_df = player_df.drop(columns = ['draft_year', 'birth_date'])

In [19]:
combine_res_df = pd.get_dummies(combine_res_df, columns = ['combine_position'], dtype=int)
#===============================================================end encoding =========================================================================

In [20]:

#====================================================Begin Standardizing ===============================

encoded_dfs = {'aggr_combine_trk': aggr_combine_trk_df, 'all_aggr_games_trk_df': all_aggr_games_trk_df, 
             'player_crr': player_crr_df, 'player': player_df, 'combine_res': combine_res_df}

scaler = StandardScaler()
aggr_combine_trk_df.head()

,nfl_id,max_speed,avg_speed,max_acceleration,avg_acceleration,min_acceleration,total_frames,drill_type_FORTY_YARD_DASH,drill_type_SHORT_SHUTTLE,drill_type_SKILL_DRILLS_DB,drill_type_SKILL_DRILLS_DL,drill_type_SKILL_DRILLS_LB,drill_type_SKILL_DRILLS_OL,drill_type_SKILL_DRILLS_TE,drill_type_SKILL_DRILLS_WR,drill_type_THREE_CONE_DRILL
0,55867,10.416667,7.966603,5.859557,1.921288,0.076739,52,1,0,0,0,0,0,0,0,0
1,55867,8.630000,4.453024,8.137175,2.873069,0.014907,797,0,0,0,0,1,0,0,0,0
2,55870,8.213333,3.117726,6.552510,2.920588,0.096148,557,0,0,0,0,0,1,0,0,0
3,55874,10.073333,7.554877,5.126226,1.827326,0.161383,54,1,0,0,0,0,0,0,0,0
4,55874,7.876667,2.914891,6.696734,2.675264,0.080069,597,0,0,0,0,0,1,0,0,0


In [21]:
columns_to_transform = aggr_combine_trk_df.columns.difference(['nfl_id'])
aggr_combine_trk_df[columns_to_transform] = scaler.fit_transform(
    aggr_combine_trk_df[columns_to_transform]
)

In [22]:
columns_to_transform = all_aggr_games_trk_df.columns.difference(['nfl_id'])
all_aggr_games_trk_df[columns_to_transform] = scaler.fit_transform(
    all_aggr_games_trk_df[columns_to_transform]
)

In [23]:
all_aggr_games_trk_df = all_aggr_games_trk_df.drop(columns = 'index')

In [24]:
player_crr_df.head()
#separate by position

,nfl_id,career_offensive_snaps,career_defensive_snaps,career_special_teams_snaps,career_games_active,career_games_started,ap_all_pro_1st_team,ap_all_pro_2nd_team,pro_bowl_original_ballot
0,55867,0,1884,92,51,44,1,0,1
1,55870,2802,0,169,47,43,0,0,0
2,55874,3222,0,167,51,49,0,1,0
3,55875,3011,0,170,51,48,0,0,0
4,55877,0,1056,387,51,2,0,0,0
